In [ ]:
import torch
import numpy as np
from typing import Optional, Tuple, Dict, Any, List
from dataclasses import dataclass
import time
from pubo_to_qubo import PUBOToQUBOReducer
import math
import gc
import os

@dataclass
class SBMConfig:
    num_steps: int = 1000           # N_s - число шагов
    num_replicas: int = 5000        # N_r - число реплик
    dt: float = 0.5                  # шаг
    a0: float = 0.1                 # a_0 параметр
    alpha: float = 0.01             # шум
    use_ternary: bool = True         # использовать дискретизацию
    threshold_scale: float = 0.7     # scale factor
    wall_position: float = 1.0        # позиция стенки
    device: str = 'cuda'           
    use_float64: bool = False         


class SimulatedBifurcationMachine:
 
    def __init__(self, J: np.ndarray, h: Optional[np.ndarray] = None,
                 device: str = 'cuda', use_float64: bool = False):
       
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        self.dtype = torch.float64 if use_float64 else torch.float32
        
        self.J = torch.tensor(J, dtype=self.dtype, device=self.device)
        self.N = self.J.shape[0]
        

        self.h = torch.tensor(h, dtype=self.dtype, device=self.device)
        
       
        mask = ~torch.eye(self.N, dtype=torch.bool, device=self.device)
        off_diag = self.J[mask].cpu().numpy()
        self.sigma = np.std(off_diag) if len(off_diag) > 0 else 1.0
        
        
    def solve(self, config: SBMConfig, verbose: bool = False):

        dt = config.dt  
        alpha = config.alpha

        q = torch.rand(config.num_replicas, self.N, 
                       dtype=self.dtype, device=self.device) * 0.5 - 0.25
        p = torch.rand(config.num_replicas, self.N, 
                       dtype=self.dtype, device=self.device) * 0.5 - 0.25

        c0 = 0.7 * config.a0 / (self.sigma * np.sqrt(self.N))
        c0 = torch.tensor(c0, dtype=self.dtype, device=self.device)
        dc = 0.01
        best_energy = float('inf')
        best_spins = None
        energies_history = []
        data_q = []
        for step in range(config.num_steps):
            t = step / config.num_steps
            a_t = config.a0 * t
#             if t > 0.9:
#                 a_t *= (1 + 10 * (t - 0.9))
                
            if config.use_ternary:
                delta_t = config.threshold_scale * t
                sign_q = torch.zeros_like(q)
                mask = torch.abs(q) > delta_t
                sign_q[mask] = torch.sign(q[mask])
            else:
                sign_q = torch.sign(q)


            local_field = 2 * torch.mm(sign_q, self.J.T) + self.h

            q += config.a0 * p * config.dt
            p -= ((config.a0 - a_t) * q - c0 * (local_field)) * config.dt

            noise = alpha * (1 - t) * torch.randn_like(p)
            p += noise
            # Стенка
            wall_mask = torch.abs(q) > config.wall_position
            q[wall_mask] = torch.sign(q[wall_mask])
            p[wall_mask] = 0.0
            
            
            #Мониторинг
            if step % 100 == 0:
                data_q.append(torch.sign(q[:, :200]).cpu())
                spins = torch.sign(q)
                energy = -torch.einsum('ij,ki,kj->k', self.J, spins, spins)
                energy -= torch.einsum('ki,i->k', spins, self.h)
                
                min_energy = energy.min().item()
                
                if min_energy < best_energy:
                    best_energy = min_energy
                    best_idx = energy.argmin().item()
                    best_spins = spins[best_idx].cpu().numpy()

                energies_history.append(min_energy)

                if verbose:
                    print(f"Step {step}, best energy: {best_energy:.4f}")

        torch.cuda.empty_cache()
        return best_spins, data_q



In [ ]:
def compute_energy_sparse(x, H1, H2, H3):
    """
    Эффективное вычисление энергии для разреженных тензоров
    """
    energy = 0.0
    n = len(x)
    
    # Линейные
    energy += np.dot(x, H1)
    
    # Квадратичные (только верхний треугольник)
    for i in range(n):
        for j in range(i+1, n):
            if H2[i, j] != 0:
                energy += H2[i, j] * x[i] * x[j]
    
    # Кубические (только i < j < k)
    for i in range(n):
        for j in range(i+1, n):
            for k in range(j+1, n):
                if H3[i, j, k] != 0:
                    energy += H3[i, j, k] * x[i] * x[j] * x[k]
    
    return energy

In [ ]:
for i in range(20, 50):
    torch.cuda.empty_cache()
    gc.collect()
    reducer = PUBOToQUBOReducer()
    H1=np.zeros(200) 
    H2=np.zeros((200, 200)) 
    H3=np.load(f"J_T_h_1/T_{i}.npy")
    J, h, pair_to_aux = reducer.reduce(H1, H2, H3, penalty= 0.5 * np.max(abs(H3)))
    
    config = SBMConfig(
        num_steps=400,
        num_replicas=5_000,
        dt=2.5,
        a0= 0.6,
        alpha=0.0001, 
        threshold_scale= 0.7,
        use_ternary=True,
        device="cuda"
        
    )
    
    solver = SimulatedBifurcationMachine(-J.toarray()/np.std(J.toarray()), -h/np.std(J.toarray()))
    spins, data_x = solver.solve(config, verbose=False)
    print(i, compute_energy_sparse(spins[:200], H1, H2, H3))
    np.save(f"solution_data_1/QUBO_SBM_{i}.npy", data_x)